# 🏭 IIoT Based Predictive Maintenance System: Cloud Analytics & Control Center

This notebook acts as the cloud-side brain of our Industrial IoT pipeline. It ingests real-time telemetry from a virtual ESP32 edge device, calculates Remaining Useful Life (RUL) using machine learning, and visualizes the data on an interactive Streamlit dashboard.

Crucially, it features a **Closed-Loop Control System**: if machine vibration crosses a critical threshold, this script automatically sends a reverse MQTT `SHUTDOWN` command to halt the edge device and prevent catastrophic failure.

### ⚙️ Step 1: Install Dependencies
First, we need to install the required Python libraries for our environment:
* `streamlit`: To build the interactive web dashboard.
* `paho-mqtt`: To handle publish/subscribe messaging with the edge device.
* `pandas` & `plotly`: For real-time time-series data visualization.
* `scikit-learn`: To power the Linear Regression model for RUL estimation.

In [ ]:
!pip install streamlit paho-mqtt pandas plotly scikit-learn -q

^C



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 🧠 Step 2: Build the Core Application (`app.py`)
Because Google Colab runs in a Jupyter environment, we cannot run Streamlit directly in a standard cell. Instead, we use the `%%writefile` magic command to save our entire Python application into a file named `app.py`.

**Key Features of this Code:**
1. **Thread-Safe MQTT Buffer:** Safely collects asynchronous data from HiveMQ without crashing the UI.
2. **Predictive Analytics:** Uses `LinearRegression` on the most recent telemetry to project exactly how many seconds remain until a machine breakdown.
3. **Automated Safety Trip:** Continuously monitors the `vib_rms` metric. If it exceeds 12.0 m/s², it immediately publishes an emergency MQTT signal back to the factory floor.

In [18]:
%%writefile app.py
import streamlit as st
import paho.mqtt.client as mqtt
import json, time, io, base64
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

st.set_page_config(page_title="IIoT Command Center", layout="wide")
st.title("🏭 IIoT Predictive Maintenance & Edge Control Center")

@st.cache_resource
def setup_mqtt_pipeline():
    buffer = []

    def on_message(client, userdata, msg):
        try:
            payload = json.loads(msg.payload.decode())
            buffer.append(payload)
        except Exception:
            pass

    try:
        client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    except AttributeError:
        client = mqtt.Client()

    client.on_message = on_message
    client.connect("broker.hivemq.com", 1883, 60)
    client.subscribe("iiot/factory/motor1/telemetry")
    client.loop_start()

    return client, buffer

client, buffer = setup_mqtt_pipeline()

if len(buffer) > 0:
    # Convert buffer to a shallow copy list(buffer) for thread safety
    df = pd.DataFrame(list(buffer)).tail(30)

    col1, col2, col3 = st.columns(3)
    latest_temp = df['temp'].iloc[-1]
    latest_vib = df['vib_rms'].iloc[-1]

    col1.info(f"**🌡️ Motor Temperature:** \n### {latest_temp:.1f} °C")
    col2.warning(f"**📳 Vibration RMS:** \n### {latest_vib:.2f} m/s²")

    if len(df) > 5:
        X = df.index.values.reshape(-1, 1)
        y = df['vib_rms'].values
        model = LinearRegression().fit(X, y)
        slope = model.coef_[0]
        rul = max(0, int((12.0 - latest_vib) / slope)) if slope > 0 else 999

        col3.success(f"**⏳ Estimated RUL:** \n### {rul} Secs")

        if latest_vib > 12.0:
            client.publish("iiot/factory/motor1/control", "SHUTDOWN")
            st.error("🚨 CRITICAL FAULT DETECTED: AUTO-SHUTDOWN SENT TO ESP32!")

    fig, ax1 = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor('#0e1117')
    ax1.set_facecolor('#0e1117')

    ax1.plot(df.index, df['temp'], color='#ff4b4b', label='Temp (°C)', linewidth=2)
    ax1.set_ylabel('Temperature (°C)', color='#ff4b4b')
    ax1.tick_params(colors='white')

    ax2 = ax1.twinx()
    ax2.plot(df.index, df['vib_rms'], color='#00d4b1', label='Vibration RMS (m/s²)', linewidth=2)
    ax2.set_ylabel('Vibration RMS (m/s²)', color='#00d4b1')
    ax2.tick_params(colors='white')

    plt.title("Real-Time Machine Telemetry Stream", color='white')
    fig.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches='tight', facecolor=fig.get_facecolor())
    buf.seek(0)
    img_str = base64.b64encode(buf.read()).decode("utf-8")
    plt.close(fig)

    st.markdown(f'<img src="data:image/png;base64,{img_str}" style="width:100%; border-radius: 8px;">', unsafe_allow_html=True)
else:
    st.info("Awaiting telemetry from Wokwi ESP32 simulation...")

time.sleep(2)
st.rerun()

Overwriting app.py


### 🚀 Step 3: Expose and Launch the Dashboard
Google Colab instances do not have open public ports. To view our Streamlit dashboard over the web, we use `localtunnel` to create a secure, temporary bridge to our notebook.

**Instructions for Launch:**
1. Run the cell below.
2. Copy the IP address printed by the `curl` command.
3. Click the `https://....loca.lt` link generated by localtunnel.
4. Paste the IP address into the **Endpoint IP** security prompt on the webpage to access your live UI.

In [ ]:
# 1. Fetch the Colab server's IP address (Your Tunnel Password)
print("Copy this IP Address to use as your Endpoint IP:")
!curl ipv4.icanhazip.com
print("\n")

# 2. Run Streamlit and forcefully bypass the npx installation prompt using -y
!streamlit run app.py & npx -y localtunnel --port 8501

Copy this IP Address to use as your Endpoint IP:
34.139.47.195


⠙

⠹⠸⠼⠴⠦your url is: https://deep-turkeys-float.loca.lt
2026-08-24 18:02:52.164 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.139.47.195:8501

